# CSI Phase And Trajectory Plots

This notebook reads the NetCDF xarray dataset produced by `processing/extract_csi_from_smb.py`, plots CSI phase in degrees for all hostnames versus cycle ID, and plots the rover trajectory in 2D.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

plt.style.use('seaborn-v0_8-whitegrid')

candidate_paths = [
    Path('csi_positions.nc'),
    Path('processing/csi_positions.nc'),
]

for candidate in candidate_paths:
    if candidate.exists():
        DATASET_PATH = candidate.resolve()
        break
else:
    raise FileNotFoundError(
        'Could not find csi_positions.nc. Run processing/extract_csi_from_smb.py first, or update candidate_paths.'
    )

ds = xr.open_dataset(DATASET_PATH)
print(f'Loaded dataset: {DATASET_PATH}')
print(ds)


In [ ]:
def tick_positions(values, max_ticks=20):
    values = np.asarray(values)
    if values.size <= max_ticks:
        return np.arange(values.size)
    return np.linspace(0, values.size - 1, max_ticks, dtype=int)


def plot_phase_heatmap(ds, experiment_id):
    experiment = ds.sel(experiment_id=experiment_id)
    csi_complex = experiment['csi_real'] + 1j * experiment['csi_imag']
    phase_deg = np.rad2deg(np.angle(csi_complex).transpose('hostname', 'cycle_id'))
    hostnames = phase_deg['hostname'].values.astype(str)
    cycle_ids = phase_deg['cycle_id'].values

    values = np.ma.masked_invalid(phase_deg.values)
    cmap = plt.get_cmap('twilight').copy()
    cmap.set_bad(color='lightgray')

    fig_width = max(10, len(cycle_ids) * 0.35)
    fig_height = max(6, len(hostnames) * 0.35)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    image = ax.imshow(
        values,
        aspect='auto',
        interpolation='none',
        cmap=cmap,
        vmin=-180,
        vmax=180,
    )

    x_ticks = tick_positions(cycle_ids)
    y_ticks = np.arange(len(hostnames))
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(cycle_ids[x_ticks], rotation=45, ha='right')
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(hostnames)
    ax.set_xlabel('Cycle ID')
    ax.set_ylabel('Hostname')
    ax.set_title(f'CSI Phase [deg] for experiment {experiment_id}')

    colorbar = fig.colorbar(image, ax=ax, pad=0.02)
    colorbar.set_label('Phase [deg]')

    fig.tight_layout()
    return fig, ax


def plot_trajectory(ds, experiment_id):
    experiment = ds.sel(experiment_id=experiment_id)
    x = experiment['rover_x'].values
    y = experiment['rover_y'].values
    cycle_ids = experiment['cycle_id'].values
    valid = np.isfinite(x) & np.isfinite(y) & (experiment['position_available'].values > 0)

    x = x[valid]
    y = y[valid]
    cycle_ids = cycle_ids[valid]

    if x.size == 0:
        raise ValueError(f'No valid rover positions for experiment {experiment_id}.')

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(x, y, '-', color='0.6', linewidth=1.5, zorder=1)
    scatter = ax.scatter(
        x,
        y,
        c=cycle_ids,
        cmap='viridis',
        s=70,
        edgecolor='black',
        zorder=2,
    )

    if cycle_ids.size <= 30:
        for xi, yi, cycle_id in zip(x, y, cycle_ids):
            ax.annotate(str(cycle_id), (xi, yi), textcoords='offset points', xytext=(5, 5), fontsize=8)

    colorbar = fig.colorbar(scatter, ax=ax, pad=0.02)
    colorbar.set_label('Cycle ID')

    ax.set_xlabel('Rover x')
    ax.set_ylabel('Rover y')
    ax.set_title(f'Rover trajectory for experiment {experiment_id}')
    ax.set_aspect('equal', adjustable='datalim')
    fig.tight_layout()
    return fig, ax


In [ ]:
experiment_ids = ds['experiment_id'].values.astype(str).tolist()
experiment_ids


In [ ]:
for experiment_id in experiment_ids:
    plot_phase_heatmap(ds, experiment_id)
    plot_trajectory(ds, experiment_id)
    plt.show()
